# Load UniProt official registry

In [5]:
import json
from pathlib import Path

In [6]:
UNIPROT_REGISTRY = Path("database_all_2026_03_02.json")

In [7]:
with UNIPROT_REGISTRY.open() as f:
    registry_data = json.load(f)

In [8]:
print(type(registry_data))
print(registry_data.keys())

<class 'dict'>
dict_keys(['results'])


In [9]:
uniprot_official_name_set = {
    entry["name"].strip().lower() for entry in registry_data["results"] if isinstance(entry, dict) and entry.get("name")
}

print("Official UniProt name count:", len(uniprot_official_name_set))

Official UniProt name count: 185


In [10]:
uniprot_official_set = {
    entry["abbrev"].strip().lower()
    for entry in registry_data["results"]
    if isinstance(entry, dict) and entry.get("abbrev")
}

print("Official UniProt abbrev count:", len(uniprot_official_set))
print("Sample:", sorted(list(uniprot_official_set))[:20])

Official UniProt abbrev count: 185
Sample: ['abcd', 'agora', 'agr', 'allergome', 'alphafolddb', 'antibodypedia', 'antifam', 'arachnoserver', 'araport', 'bgee', 'bindingdb', 'biocyc', 'biogrid', 'biogrid-orcs', 'biomuta', 'bmrb', 'brenda', 'carbonyldb', 'card', 'cazy']


## Load BERDL prefixes.txt

In [11]:
BERDL_PREFIXES = Path("prefixes.txt")

In [12]:
berdl_set = set()

In [13]:
with BERDL_PREFIXES.open() as f:
    for line in f:
        berdl_set.add(line.strip().lower())

print("BERDL idmapping prefixes:", len(berdl_set))

BERDL idmapping prefixes: 103


## Load parquet prefixes 

In [14]:
from pyspark.sql.functions import lower, col

In [15]:
from pyspark.sql import SparkSession

In [16]:
spark = SparkSession.builder.appName("PrefixExploration").getOrCreate()

In [28]:
df = spark.read.parquet("part-00000-0a0d0261-1fee-477d-90d8-1df048058fbf-c000.snappy.parquet")

In [30]:
df.printSchema()

root
 |-- entity_id: string (nullable = true)
 |-- db: string (nullable = true)
 |-- xref: string (nullable = true)
 |-- description: string (nullable = true)
 |-- _dlt_load_id: string (nullable = true)
 |-- _dlt_id: string (nullable = true)
 |-- relationship: string (nullable = true)



In [18]:
parquet_set = {row["db"] for row in df.select(lower(col("db")).alias("db")).distinct().collect()}

print("Parquet prefixes:", len(parquet_set))

Parquet prefixes: 82


In [19]:
## Prefixes in parquet but not in UniProt official list
parquet_not_in_uniprot = parquet_set - uniprot_official_set

In [20]:
print(len(parquet_not_in_uniprot))
print(sorted(list(parquet_not_in_uniprot)))

3
['ec', 'ncbitaxon', 'uniprot']


### Interpretation

These are not true registry gaps:

- **ec** – Represents EC numbers. 
- **ncbitaxon** – A naming variation of NCBI Taxonomy.
- **uniprot** – UniProt itself is not listed as an external cross-reference database.

Conclusion: No external databases detected


In [21]:
## Prefixes in BERDL idmapping but not in UniProt official list
berdl_not_in_uniprot = berdl_set - uniprot_official_set

In [22]:
print(len(berdl_not_in_uniprot))
print(sorted(list(berdl_not_in_uniprot)))

25
['crc64', 'embl-cds', 'ensembl_pro', 'ensembl_trs', 'ensemblgenome', 'ensemblgenome_pro', 'ensemblgenome_trs', 'gene_name', 'gene_orderedlocusname', 'gene_orfname', 'gene_synonym', 'gi', 'ncbi_taxid', 'nextprot', 'pharmgkb', 'refseq_nt', 'treefam', 'uniparc', 'uniprotkb-id', 'uniref100', 'uniref50', 'uniref90', 'wbparasite_trs_pro', 'wormbase_pro', 'wormbase_trs']


### Classification of Differences

### Classification of BERDL Prefixes Not Present in UniProt Official Cross-Reference Registry

The following prefixes appear in the BERDL idmapping-derived set but are not listed in the UniProt official cross-reference registry.

They fall into several categories:

    1.	Internal UniProt metadata
	2.	Subtype mappings 
	3.	External biological databases
	4.	Deprecated or taxonomy identifiers
	5.	UniProt-derived resources


In [76]:
registry_data["results"][:1]

[{'name': 'ABCD curated depository of sequenced antibodies',
  'id': 'DB-0236',
  'abbrev': 'ABCD',
  'linkType': 'Explicit',
  'servers': ['https://web.expasy.org/abcd'],
  'dbUrl': 'https://web.expasy.org/cgi-bin/abcd/search_abcd.pl?input=%u',
  'category': 'Protocols and materials databases',
  'statistics': {'reviewedProteinCount': 3196, 'unreviewedProteinCount': 619}}]

In [77]:
from collections import defaultdict

In [78]:
## create a dictionary with empty lists

classification = defaultdict(list)

In [79]:
SUBTYPE_MAPPING = {
    "embl-cds",  # EMBL CDS subtype
    "refseq_nt",  # RefSeq nucleotide subtype
}

for p in sorted(berdl_not_in_uniprot):
    # internal annotation fields
    if p.startswith("gene_") or p in {"crc64", "uniprotkb-id"}:
        classification["internal_metadata"].append(p)

    # UniProt derived databases
    elif p.startswith("uniref") or p == "uniparc":
        classification["uniprot_derived_db"].append(p)

    # deprecated identifiers
    elif p in {"gi"}:
        classification["deprecated_identifier"].append(p)

    # taxonomy identifiers
    elif p in {"ncbi_taxid"}:
        classification["taxonomy_identifier"].append(p)

    # subtype-specific identifiers
    elif p in {"embl-cds", "refseq_nt"} or any(token in p for token in ["_pro", "_trs"]):
        classification["subtype_mapping"].append(p)

    # external database candidate
    else:
        classification["external_database_candidate"].append(p)

In [80]:
for k, v in classification.items():
    print(f"\n{k} ({len(v)}):")
    print(sorted(v))


internal_metadata (6):
['crc64', 'gene_name', 'gene_orderedlocusname', 'gene_orfname', 'gene_synonym', 'uniprotkb-id']

subtype_mapping (9):
['embl-cds', 'ensembl_pro', 'ensembl_trs', 'ensemblgenome_pro', 'ensemblgenome_trs', 'refseq_nt', 'wbparasite_trs_pro', 'wormbase_pro', 'wormbase_trs']

external_database_candidate (4):
['ensemblgenome', 'nextprot', 'pharmgkb', 'treefam']

deprecated_identifier (1):
['gi']

taxonomy_identifier (1):
['ncbi_taxid']

uniprot_derived_db (4):
['uniparc', 'uniref100', 'uniref50', 'uniref90']


In [81]:
external_candidates = classification["external_database_candidate"]

In [82]:
for p in external_candidates:
    in_name = p in uniprot_official_name_set
    in_abbrev = p in uniprot_official_set
    count = df.filter(lower(col("db")) == p).count()
    print(f"{p:20} | name={in_name} | abbrev={in_abbrev}")
    print(f"{p:20} | parquet_rows={count}")

ensemblgenome        | name=False | abbrev=False
ensemblgenome        | parquet_rows=0
nextprot             | name=False | abbrev=False
nextprot             | parquet_rows=0
pharmgkb             | name=False | abbrev=False
pharmgkb             | parquet_rows=0
treefam              | name=False | abbrev=False
treefam              | parquet_rows=0


Some prefixes classified as external database candidates (e.g., nextprot, pharmgkb, treefam) do not currently appear in the BERDL parquet dataset.

This indicates that while these namespaces correspond to real biological databases, they are not used in the current dataset snapshot. They remain classified as external databases based on their semantic meaning rather than dataset usage.

### A. UniProt annotation metadata 
- crc64
- gene_name
- gene_orderedlocusname
- gene_orfname
- gene_synonym
- uniprotkb-id

These fields represent internal UniProt annotations rather than cross-references to external databases. Examples include gene name annotations and sequence checksums maintained directly within UniProt records.


### B. UniProt derived databases 
- uniparc 
- uniref100
- uniref50
- uniref90

These are UniProt internal resources, no need remapping. 


### C. Internal NCBI identifiers
- gi
- ncbi_taxid
  
ncbi_taxid is taxonomy identifier,
gi used by NCBI but has been officially deprecated.



### D. Database subtype mappings 
- embl-cds
- refseq_nt
- ensembl_pro
- ensembl_trs
- ensemblgenome_pro
- ensemblgenome_trs
- wormbase_pro
- wormbase_trs
- wbparasite_trs_pro

#### patterns:
	•	_pro → protein identifiers
	•	_trs → transcript identifiers
	•	_cds → coding sequence identifiers
	•	_nt → nucleotide accessions

These indicate the identifier type within a parent database. Need normalize to parent database prefix.


### E. External database 

- ensemblgenome
- nextprot
- pharmgkb
- treefam


#### examples:

	•	EnsemblGenome – genome annotation database
	•	NextProt – human protein knowledgebase
	•	PharmGKB – pharmacogenomics database
	•	TreeFam – phylogenetic gene family database